In [1]:
import os
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyvisa as pv

import labmate
from labmate.acquisition_notebook import AcquisitionAnalysisManager

from utils.naming import make_run_dir

# Setup Directory

In [6]:
DATA_DIR = "data/spectrumanalyser"
os.makedirs(DATA_DIR, exist_ok=True)

In [7]:
# FOLDER
SAMPLE = "pda36a"
MEAS = "spectrum_trace"
ACQ_CELL = f"{MEAS}_{SAMPLE}"

RUN_DIR = make_run_dir(data_dir=DATA_DIR, meas=MEAS, sample=SAMPLE)

print("Spectrum Analyser data will be saved in:", RUN_DIR)

Saving data to: data/spectrumanalyser\spectrum_trace\2026-08-04_pda36a_001
Spectrum Analyser data will be saved in: data/spectrumanalyser\spectrum_trace\2026-08-04_pda36a_001


# Connect to Spectrum Analyser

In [8]:
rm = pv.ResourceManager()
print(rm.list_resources())

('USB0::0x0957::0x0A0B::MY50200339::INSTR', 'ASRL1::INSTR', 'ASRL4::INSTR')


In [10]:
sa = rm.open_resource('USB0::0x0957::0x0A0B::MY50200339::INSTR') # Spectrum Analyser
print(sa.query('*IDN?'))

Agilent Technologies,N9020A,MY50200339,A.08.03



# Acquisition

## Basic Trace Query

In [11]:
print(sa.query(":INST:SEL?"))

SA



In [13]:
traces = {}

sa.write(":FORM:DATA ASC")

for i in range(1, 7):
    try:
        traces[f"TRACE{i}"] = sa.query_ascii_values(f":TRAC:DATA? TRACE{i}")
    except Exception:
        pass

print(traces.keys())

dict_keys(['TRACE1', 'TRACE2', 'TRACE3', 'TRACE4', 'TRACE5', 'TRACE6'])


In [14]:
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(ACQ_CELL)

traces = {}

sa.write(":FORM:DATA ASC")

for i in range(1, 7):
    try:
        traces[f"TRACE{i}"] = sa.query_ascii_values(f":TRAC:DATA? TRACE{i}")
    except Exception:
        pass

aqm.save_acquisition(traces=traces)

INFO:1:2026_08_04__16_57_30__spectrum_trace_pda36a
